# Implementation obnoxious facility location problem
This document contains the cleaned up code for the addition proposed in the Bachelor End Project of Lucas Klaasse. 

# Backend definitions

## Initial imports
These packages are used in the definitions that follow.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import math
from pyscipopt import Model, quicksum

## General definitions
The following definitions are general definitions or implementations of well known formulas

In [ ]:
def solve_abc_form(a,b,c) -> list:
    '''
    This function is an implementation of the abc formula for solving x in a*x^2 + b*x + c = 0.
    :param a: a
    :param b: b
    :param c: c
    '''
    D = (b ** 2) - (4 * a * c) 
    if D < 0:
        return []
    else:
        return [float((- b + np.sqrt(D)) / (2 * a)), float(( - b - np.sqrt(D)) / (2 * a))]



def interior_circle(point : list, circle : list) -> bool:
    '''
    This function looks whether or not a point is inside the circle.
    :param point: The point that we want to check.
    :param circle: The circle that we want to check.
    '''
    if (circle[0] - point[0]) ** 2 + (circle[1] - point[1]) ** 2 < circle[2] ** 2 - 1e-4:
        return True
    else:
        return False


def generate_points(x_lb : float, x_ub : float, y_lb : float, y_ub : float) -> list:
    '''
    A function that creates points inside the box for testing the implementation of the convex hull creation with the LP.
    :param x_lb: The lowest x of the box.
    :param x_ub: The highest x of the box.
    :param y_lb: The lowest y of the box.
    :param y_ub: The highest y of the box.
    '''
    points = []
    tol = math.ceil(x_ub - x_lb)
    x = np.linspace(x_lb - tol, x_ub + tol, tol * 5)
    y = np.linspace(y_lb - tol, y_ub + tol, tol * 5)
    for i in x:
        for j in y:
            points.append([i,j])
    return points

## Calculating intersections
The following definitions are for calculating intersections in the main program of the code

In [ ]:
def box_circles(x_lb, x_ub, y_lb, y_ub, circle) -> list:
    '''
    This function takes a box and a circle and returns the intersections that this circle has with the box.
    :param x_lb: The lowest x of the box.
    :param x_ub: The highest x of the box.
    :param y_lb: The lowest y of the box.
    :param y_ub: The highest y of the box.
    :param circle: The circle that we are considering
    '''
    x, y, r = circle
    intersections = []

    # We first check which sides the circle intersects with
    if x - r <= x_lb <= x + r:
        a = 1
        b = -2 * y
        c = x ** 2 - 2 * x * x_lb + x_lb ** 2 + y ** 2 - r ** 2
        py = solve_abc_form(a,b,c)
        if py != []:
            for yi in py:
                if ((x_lb, yi) not in intersections) and y_lb <= yi <= y_ub:
                    intersections.append([x_lb, yi])
    
    if x - r <= x_ub <= x + r:
        a = 1
        b = -2 * y
        c = x ** 2 - 2 * x * x_ub + x_ub ** 2 + y ** 2 - r ** 2
        py = solve_abc_form(a,b,c)
        if py != []:
            for yi in py:
                if ((x_ub, yi) not in intersections) and y_lb <= yi <= y_ub:
                    intersections.append([x_ub, yi])
    
    if y - r <= y_lb <= y + r:
        a = 1
        b = -2 * x
        c = y ** 2 - 2 * y * y_lb + y_lb ** 2 + x ** 2 - r ** 2
        px = solve_abc_form(a,b,c)
        if px != []:
            for xi in px:
                if ((xi, y_lb) not in intersections) and y_lb <= xi <= y_ub:
                    intersections.append([xi, y_lb])
            
    if y - r <= y_ub <= y + r:
        a = 1
        b = -2 * x
        c = y ** 2 - 2 * y * y_ub + y_ub ** 2 + x ** 2 - r ** 2
        px = solve_abc_form(a,b,c)
        if px != []:
            for xi in px:
                if ((xi, y_ub) not in intersections) and y_lb <= xi <= y_ub:
                    intersections.append([xi, y_ub])
    
    return intersections





def circles_circles(c1, c2) -> list:
    '''
    This function takes two circles and returns there intersection points.
    :param c1: The first circle considered.
    :param c2: The second circle considered.
    '''
    x0, y0, r0 = c1
    x1, y1, r1 = c2

    dx = x1 - x0
    dy = y1 - y0
    dist = np.sqrt(dx**2 + dy**2)

    # No intersection
    if dist > r0 + r1:
        return []

    # One circle completely inside the other
    if dist < abs(r0 - r1):
        return []

    # Same centre
    if dist == 0:
        return []

    # Distance from centre 0 to midpoint between intersections
    a = (r0**2 - r1**2 + dist**2) / (2 * dist)

    h_squared = r0**2 - a**2
    h_squared = max(0, h_squared)
    h = np.sqrt(h_squared)

    # Midpoint
    xm = x0 + a * dx / dist
    ym = y0 + a * dy / dist

    # Offset from midpoint
    rx = -dy * h / dist
    ry = dx * h / dist

    p1 = [float(xm + rx), float(ym + ry)]
    p2 = [float(xm - rx), float(ym - ry)]
    
    # Tangent circles: only one intersection
    if np.isclose(h, 0):
        return [p1]

    return [p1, p2]

## LP definitions
These definitions are there to create the convex hull of the set of points

In [ ]:
def create_conv_region(points: list, solutions: list) -> list:
    '''
    For each solution, find a cutting plane that separates
    the solution from all points.
    :param points: The points that need to be on one side of the cutting plane.
    :param solutions: The solutions of the program that we want to try to cut.
    '''
    cutting_planes = []
    t = 0

    for s in solutions:
        model = Model("CuttingPlane")
        model.hideOutput()

        # Variables
        p0 = model.addVar("p0", lb=-1, ub=1)
        p1 = model.addVar("p1", lb=-1, ub=1)
        p2 = model.addVar("p2", lb=-1, ub=1)

        # Objective: maximize p0*s[0] + p1*s[1] + p2
        model.setObjective(
            p0 * s[0] + p1 * s[1] + p2,
            sense="maximize"
        )

        # Constraints: p0*x + p1*y + p2 <= 0
        for x, y in points:
            model.addCons(p0*x + p1*y + p2 <= 0)

        # Solve
        model.optimize()

        if model.getStatus() == "optimal":
            plane = [
                model.getVal(p0),
                model.getVal(p1),
                model.getVal(p2)
            ]

            if model.getObjVal() >= 1e-4 and plane not in cutting_planes:
                cutting_planes.append(plane)
                print(f"{s} at t = {t}, {plane}")
                t += 1

    return cutting_planes

## GS definitions
